# APG&E Credit Threshold Change Analysis

**Event:** APG&E raised its credit threshold from **500 → 600** on 3/23 morning.

**Impact:** Sessions with credit scores 500-599 that previously passed APG&E credit check now fail, directly hitting **Conversion After Credit** and flowing through to Cart VC and all-in VC.

---

| Section | Description |
|---------|-------------|
| **1. Impact to Performance** | All-in funnel WoW movement + isolated impact on 500-599 APG&E band |
| **2. Counterfactual Analysis** | How much of the VC decline is explained by this threshold change? |
| **3. TLDR** | One-paragraph summary with key numbers |

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import certifi
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from datetime import date
from dotenv import load_dotenv
from databricks import sql as databricks_sql
from IPython.display import display, Markdown

load_dotenv(override=True)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

In [2]:
# ── Config ──
PRE_START, PRE_END = date(2026, 3, 16), date(2026, 3, 22)
POST_START, POST_END = date(2026, 3, 23), date(2026, 3, 29)
DEFAULT_CHANNELS = ['Paid Search', 'Direct', 'Organic', 'pMax']

def safe_rate(num, den):
    return num / den if den > 0 else 0.0

def pct_change(curr, prior):
    return (curr - prior) / prior if prior != 0 else float('nan')

def fmt_pct(val):
    return f'{val * 100:.2f}%'

def fmt_pct_ch(val):
    return f'{val * 100:+.1f}%'

def credit_band(score):
    if pd.isna(score): return 'No Credit Run'
    s = int(score)
    if s < 500: return '<500'
    elif s < 600: return '500-599'
    elif s < 700: return '600-699'
    elif s < 800: return '700-799'
    else: return '800+'

## Load Data

In [3]:
_HOST = os.getenv('DATABRICKS_HOST', '')
_TOKEN = os.getenv('DATABRICKS_TOKEN', '')
_HTTP_PATH = os.getenv('DATABRICKS_HTTP_PATH', '')

query_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'session_level_query'))
with open(query_path) as f:
    raw = f.read()
raw = raw.replace('%sql', '', 1).strip()
raw = raw.replace("('2026-01-01')::date AS start_date", f"('{PRE_START}')::date AS start_date")
raw = raw.replace("(CURRENT_DATE)::date AS end_date", f"('{POST_END}')::date AS end_date")

conn = databricks_sql.connect(
    server_hostname=_HOST.replace('https://', '').strip('/'),
    http_path=_HTTP_PATH,
    access_token=_TOKEN,
)
cursor = conn.cursor()
cursor.execute(raw)
rows = cursor.fetchall()
cols = [desc[0] for desc in cursor.description]
conn.close()

df = pd.DataFrame(rows, columns=cols)
print(f'Loaded {len(df):,} sessions')

Loaded 249,958 sessions


In [4]:
# ── Type normalization ──
if 'session_start_date_est' in df.columns:
    df['session_start_date_est'] = pd.to_datetime(df['session_start_date_est']).dt.date

for col in ['cart_order', 'phone_order']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
df['total_order'] = df.get('cart_order', 0) + df.get('phone_order', 0)

flag_cols = ['session', 'zip_entry', 'has_cart', 'cart_order', 'cart_ssn_done',
             'gross_call', 'queue_call', 'phone_order', 'cart_credit_fail',
             'cart_provider_pass', 'cart_volt_fail', 'cart_qual_fail', 'total_order']
for col in flag_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

if 'first_run_credit_score' in df.columns:
    df['first_run_credit_score'] = pd.to_numeric(df['first_run_credit_score'], errors='coerce')

# ── Derived columns ──
df = df[df['marketing_channel'].isin(DEFAULT_CHANNELS)].copy()
df['period'] = df['session_start_date_est'].apply(
    lambda d: 'Pre (3/16-3/22)' if d <= PRE_END else 'Post (3/23-3/29)'
)
df['credit_band'] = df['first_run_credit_score'].apply(credit_band)
df['is_apge'] = df['first_run_provider_name'].fillna('').str.upper().str.contains('APG')

pre = df[df['period'] == 'Pre (3/16-3/22)']
post = df[df['period'] == 'Post (3/23-3/29)']
print(f'Pre period: {len(pre):,} sessions | Post period: {len(post):,} sessions')

Pre period: 63,869 sessions | Post period: 63,462 sessions


---
## 1. Impact to Performance

### All-in Funnel KPIs (WoW)

In [ ]:
# ── DEBUG: Inspect raw column values to validate KPI formulas ──

print("=== Column value checks ===\n")

for col in ['session', 'has_cart', 'cart_ssn_done', 'cart_order', 'phone_order', 'total_order']:
    if col in df.columns:
        print(f"{col}:")
        print(f"  dtype: {df[col].dtype}")
        print(f"  unique values (sample): {sorted(df[col].dropna().unique()[:10])}")
        print(f"  value_counts (top 5):\n{df[col].value_counts().head()}")
        print(f"  sum: {df[col].sum():,.0f}")
        print()

print("\n=== Raw KPI numerators and denominators ===\n")
for label, num_col, den_col in kpis:
    num_pre = pre[num_col].sum()
    den_pre = pre[den_col].sum()
    num_post = post[num_col].sum()
    den_post = post[den_col].sum()
    print(f"{label}:")
    print(f"  Pre:  {num_col}={num_pre:,.0f} / {den_col}={den_pre:,.0f} = {safe_rate(num_pre, den_pre)*100:.4f}%")
    print(f"  Post: {num_col}={num_post:,.0f} / {den_col}={den_post:,.0f} = {safe_rate(num_post, den_post)*100:.4f}%")
    print()

In [5]:
kpis = [
    ('Cart RR',              'has_cart',      'session'),
    ('SSN Submit Rate',      'cart_ssn_done', 'has_cart'),
    ('Conv After Credit',    'cart_order',    'cart_ssn_done'),
    ('Cart Conversion',      'cart_order',    'has_cart'),
    ('Cart VC',              'cart_order',    'session'),
    ('Phone VC',             'phone_order',   'session'),
    ('VC (Total)',           'total_order',   'session'),
]

funnel_rows = []
for label, num_col, den_col in kpis:
    r_pre = safe_rate(pre[num_col].sum(), pre[den_col].sum())
    r_post = safe_rate(post[num_col].sum(), post[den_col].sum())
    ch = pct_change(r_post, r_pre)
    funnel_rows.append({
        'KPI': label,
        'Pre Week': fmt_pct(r_pre),
        'Post Week': fmt_pct(r_post),
        '% Change': fmt_pct_ch(ch),
        'pp Change': f'{(r_post - r_pre) * 100:+.2f}pp',
    })

funnel_summary = pd.DataFrame(funnel_rows)
display(funnel_summary.style.hide(axis='index'))

KPI,Pre Week,Post Week,% Change,pp Change
Cart RR,21.70%,21.36%,-1.6%,-0.34pp
SSN Submit Rate,28.06%,34.55%,+23.1%,+6.49pp
Conv After Credit,55.93%,46.75%,-16.4%,-9.17pp
Cart Conversion,15.69%,16.15%,+2.9%,+0.46pp
Cart VC,3.41%,3.45%,+1.3%,+0.05pp
Phone VC,1.57%,1.67%,+6.5%,+0.10pp
VC (Total),4.97%,5.12%,+3.0%,+0.15pp


### Isolating the Affected Population: APG&E First-Run, Credit Score 500-599

In [6]:
affected_pre = pre[(pre['credit_band'] == '500-599') & (pre['is_apge']) & (pre['cart_ssn_done'] == 1)]
affected_post = post[(post['credit_band'] == '500-599') & (post['is_apge']) & (post['cart_ssn_done'] == 1)]

n_pre = len(affected_pre)
n_post = len(affected_post)
orders_pre = affected_pre['cart_order'].sum()
orders_post = affected_post['cart_order'].sum()
conv_pre = safe_rate(orders_pre, n_pre)
conv_post = safe_rate(orders_post, n_post)

cf_pre = safe_rate(affected_pre['cart_credit_fail'].sum(), n_pre)
cf_post = safe_rate(affected_post['cart_credit_fail'].sum(), n_post)

affected_summary = pd.DataFrame([
    {'Metric': 'SSN Sessions', 'Pre Week': f'{int(n_pre):,}', 'Post Week': f'{int(n_post):,}'},
    {'Metric': 'Cart Orders', 'Pre Week': f'{int(orders_pre):,}', 'Post Week': f'{int(orders_post):,}'},
    {'Metric': 'Conv After Credit', 'Pre Week': fmt_pct(conv_pre), 'Post Week': fmt_pct(conv_post)},
    {'Metric': 'Conv After Credit % Change', 'Pre Week': '', 'Post Week': fmt_pct_ch(pct_change(conv_post, conv_pre)) if n_pre > 0 and n_post > 0 else 'N/A'},
    {'Metric': 'Credit Fail Rate', 'Pre Week': fmt_pct(cf_pre), 'Post Week': fmt_pct(cf_post)},
])

display(Markdown('**APG&E first-run credit check, score 500-599, SSN-submitted sessions:**'))
display(affected_summary.style.hide(axis='index'))

**APG&E first-run credit check, score 500-599, SSN-submitted sessions:**

Metric,Pre Week,Post Week
SSN Sessions,30,23
Cart Orders,25,4
Conv After Credit,83.33%,17.39%
Conv After Credit % Change,,-79.1%
Credit Fail Rate,0.00%,100.00%


### Conversion After Credit by Credit Band (APG&E First-Run)

In [7]:
apge_ssn = df[(df['is_apge']) & (df['cart_ssn_done'] == 1)]

band_data = apge_ssn.groupby(['period', 'credit_band']).agg(
    ssn_sessions=('cart_ssn_done', 'sum'),
    orders=('cart_order', 'sum'),
).reset_index()
band_data['Conv After Credit'] = band_data.apply(lambda r: safe_rate(r['orders'], r['ssn_sessions']), axis=1)

band_pivot = band_data.pivot_table(
    index='credit_band', columns='period',
    values=['ssn_sessions', 'orders', 'Conv After Credit'],
    aggfunc='first'
).reindex(['<500', '500-599', '600-699', '700-799', '800+'])

display(band_pivot.fillna(0).style.format({
    ('ssn_sessions', 'Pre (3/16-3/22)'): '{:,.0f}',
    ('ssn_sessions', 'Post (3/23-3/29)'): '{:,.0f}',
    ('orders', 'Pre (3/16-3/22)'): '{:,.0f}',
    ('orders', 'Post (3/23-3/29)'): '{:,.0f}',
    ('Conv After Credit', 'Pre (3/16-3/22)'): '{:.2%}',
    ('Conv After Credit', 'Post (3/23-3/29)'): '{:.2%}',
}))

In [8]:
# Visualize the conversion shift by credit band
band_chart = band_data[band_data['credit_band'].isin(['<500', '500-599', '600-699', '700-799', '800+'])].copy()
band_chart['Conv After Credit (%)'] = band_chart['Conv After Credit'] * 100

fig_band = px.bar(
    band_chart, x='credit_band', y='Conv After Credit (%)', color='period',
    barmode='group', title='APG&E Conv After Credit by Credit Band — Pre vs Post Threshold Change',
    labels={'credit_band': 'Credit Score Band', 'Conv After Credit (%)': 'Conversion After Credit (%)'},
    color_discrete_map={'Pre (3/16-3/22)': '#3498db', 'Post (3/23-3/29)': '#e74c3c'},
    category_orders={'credit_band': ['<500', '500-599', '600-699', '700-799', '800+']},
)
fig_band.update_layout(height=400)
fig_band.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

### Size of Affected Population

In [9]:
total_ssn_pre = pre['cart_ssn_done'].sum()
total_ssn_post = post['cart_ssn_done'].sum()

size_summary = pd.DataFrame([
    {'': 'Total SSN sessions', 'Pre Week': f'{int(total_ssn_pre):,}', 'Post Week': f'{int(total_ssn_post):,}'},
    {'': 'APG&E 500-599 SSN sessions', 'Pre Week': f'{int(n_pre):,}', 'Post Week': f'{int(n_post):,}'},
    {'': '% of total SSN', 'Pre Week': f'{safe_rate(n_pre, total_ssn_pre)*100:.1f}%', 'Post Week': f'{safe_rate(n_post, total_ssn_post)*100:.1f}%'},
])
display(size_summary.style.hide(axis='index'))

,Pre Week,Post Week
Total SSN sessions,"3,889","4,684"
APG&E 500-599 SSN sessions,30,23
% of total SSN,0.8%,0.5%


---
## 2. Counterfactual Analysis

**Question:** If APG&E 500-599 sessions in the post week had maintained their pre-week Conversion After Credit rate, how many more orders would we have — and how much of the total VC decline does this explain?

In [10]:
counterfactual_orders = max(0, (conv_pre - conv_post) * n_post)

total_orders_post = post['total_order'].sum()
total_sessions_post = post['session'].sum()
cart_orders_post = post['cart_order'].sum()

vc_post = safe_rate(total_orders_post, total_sessions_post)
vc_cf = safe_rate(total_orders_post + counterfactual_orders, total_sessions_post)
cart_vc_post = safe_rate(cart_orders_post, total_sessions_post)
cart_vc_cf = safe_rate(cart_orders_post + counterfactual_orders, total_sessions_post)

total_orders_pre = pre['total_order'].sum()
total_sessions_pre = pre['session'].sum()
vc_pre = safe_rate(total_orders_pre, total_sessions_pre)
vc_delta = vc_post - vc_pre
apge_cost = vc_cf - vc_post
pct_explained = (apge_cost / abs(vc_delta)) * 100 if vc_delta != 0 else 0

cf_results = pd.DataFrame([
    {'Metric': 'Estimated lost orders', 'Value': f'{counterfactual_orders:,.1f}'},
    {'Metric': '', 'Value': ''},
    {'Metric': 'Actual post-week Cart VC', 'Value': fmt_pct(cart_vc_post)},
    {'Metric': 'Counterfactual Cart VC', 'Value': fmt_pct(cart_vc_cf)},
    {'Metric': 'APG&E threshold cost to Cart VC', 'Value': f'{(cart_vc_cf - cart_vc_post) * 100:+.3f}pp'},
    {'Metric': '', 'Value': ''},
    {'Metric': 'Actual post-week VC', 'Value': fmt_pct(vc_post)},
    {'Metric': 'Counterfactual VC', 'Value': fmt_pct(vc_cf)},
    {'Metric': 'APG&E threshold cost to VC', 'Value': f'{(vc_cf - vc_post) * 100:+.3f}pp'},
    {'Metric': '', 'Value': ''},
    {'Metric': 'Total VC WoW change', 'Value': f'{vc_delta * 100:+.3f}pp'},
    {'Metric': '% of VC decline explained by APG&E', 'Value': f'{pct_explained:.1f}%'},
])
display(cf_results.style.hide(axis='index'))

Metric,Value
Estimated lost orders,15.2
,
Actual post-week Cart VC,3.45%
Counterfactual Cart VC,3.47%
APG&E threshold cost to Cart VC,+0.024pp
,
Actual post-week VC,5.12%
Counterfactual VC,5.15%
APG&E threshold cost to VC,+0.024pp
,


In [12]:
# Waterfall: VC decomposition
remaining = vc_delta - apge_cost

fig_cf = go.Figure(go.Waterfall(
    x=['Pre-Week VC', 'APG&E Threshold Impact', 'Other Factors', 'Post-Week VC'],
    y=[vc_pre * 100, apge_cost * 100, remaining * 100, vc_post * 100],
    measure=['absolute', 'relative', 'relative', 'total'],
    text=[fmt_pct(vc_pre), f'{apge_cost*100:+.3f}pp', f'{remaining*100:+.3f}pp', fmt_pct(vc_post)],
    textposition='outside',
    connector={'line': {'color': 'gray', 'width': 1}},
    increasing={'marker': {'color': '#2ecc71'}},
    decreasing={'marker': {'color': '#e74c3c'}},
    totals={'marker': {'color': '#3498db'}},
))
fig_cf.update_layout(
    title='VC WoW Bridge: APG&E Threshold Impact vs Other Factors',
    yaxis_title='VC (%)', height=420,
)
fig_cf.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

### Provider Mix Check
Did the credit threshold change shift volume across providers?

In [13]:
provider_pre = pre[pre['cart_ssn_done'] == 1].groupby('first_run_provider_name').agg(
    pre_sessions=('cart_ssn_done', 'sum'), pre_orders=('cart_order', 'sum')
).reset_index()
provider_post = post[post['cart_ssn_done'] == 1].groupby('first_run_provider_name').agg(
    post_sessions=('cart_ssn_done', 'sum'), post_orders=('cart_order', 'sum')
).reset_index()

providers = pd.merge(provider_pre, provider_post, on='first_run_provider_name', how='outer').fillna(0)
providers['Pre Conv'] = providers.apply(lambda r: safe_rate(r['pre_orders'], r['pre_sessions']), axis=1)
providers['Post Conv'] = providers.apply(lambda r: safe_rate(r['post_orders'], r['post_sessions']), axis=1)
providers = providers.sort_values('post_sessions', ascending=False).head(10)

providers_display = providers.copy()
providers_display['pre_sessions'] = providers_display['pre_sessions'].apply(lambda x: f'{int(x):,}')
providers_display['post_sessions'] = providers_display['post_sessions'].apply(lambda x: f'{int(x):,}')
providers_display['pre_orders'] = providers_display['pre_orders'].apply(lambda x: f'{int(x):,}')
providers_display['post_orders'] = providers_display['post_orders'].apply(lambda x: f'{int(x):,}')
providers_display['Pre Conv'] = providers_display['Pre Conv'].apply(fmt_pct)
providers_display['Post Conv'] = providers_display['Post Conv'].apply(fmt_pct)
providers_display.columns = ['Provider', 'Pre SSN Sessions', 'Pre Orders', 'Post SSN Sessions', 'Post Orders', 'Pre Conv After Credit', 'Post Conv After Credit']
display(providers_display.style.hide(axis='index'))

Provider,Pre SSN Sessions,Pre Orders,Post SSN Sessions,Post Orders,Pre Conv After Credit,Post Conv After Credit
4Change Energy,"1,159",616,"1,366",705,53.15%,51.61%
Gexa Energy,709,371,"1,009",576,52.33%,57.09%
APG&E,397,269,532,368,67.76%,69.17%
TXU Energy,128,66,201,115,51.56%,57.21%
Frontier Utilities,170,88,169,87,51.76%,51.48%
Cirro,117,54,160,73,46.15%,45.62%
Discount Power,158,90,153,69,56.96%,45.10%
Rhythm Energy,258,137,107,51,53.10%,47.66%
Express Energy,106,39,95,44,36.79%,46.32%
Constellation,16,10,51,34,62.50%,66.67%


---
## 3. TLDR

In [14]:
direction = 'declined' if conv_post < conv_pre else 'improved'
conv_ch = pct_change(conv_post, conv_pre)

tldr = f"""
**APG&E raised its credit threshold from 500 to 600 on 3/23.**

The 500-599 credit band for APG&E first-run credit checks saw **Conversion After Credit 
{direction} {fmt_pct_ch(conv_ch)}** WoW (from {fmt_pct(conv_pre)} to {fmt_pct(conv_post)}).

This population represents **~{safe_rate(n_post, total_ssn_post)*100:.1f}%** of all SSN-submitted 
sessions. The estimated cost is **~{counterfactual_orders:,.0f} lost cart orders** in the post week.

**Impact on all-in KPIs:**
- Cart VC: **{(cart_vc_cf - cart_vc_post)*100:+.3f}pp** drag
- VC: **{(vc_cf - vc_post)*100:+.3f}pp** drag

This explains approximately **{pct_explained:.0f}%** of the total WoW VC movement 
({vc_delta*100:+.3f}pp).

Sessions in this credit band that were previously converting through APG&E are now failing 
credit check. Some may re-run with another provider, but the immediate hit to Conversion 
After Credit is mechanical — these sessions can no longer qualify for APG&E plans.
"""

display(Markdown(tldr))


**APG&E raised its credit threshold from 500 to 600 on 3/23.**

The 500-599 credit band for APG&E first-run credit checks saw **Conversion After Credit 
declined -79.1%** WoW (from 83.33% to 17.39%).

This population represents **~0.5%** of all SSN-submitted 
sessions. The estimated cost is **~15 lost cart orders** in the post week.

**Impact on all-in KPIs:**
- Cart VC: **+0.024pp** drag
- VC: **+0.024pp** drag

This explains approximately **16%** of the total WoW VC movement 
(+0.147pp).

Sessions in this credit band that were previously converting through APG&E are now failing 
credit check. Some may re-run with another provider, but the immediate hit to Conversion 
After Credit is mechanical — these sessions can no longer qualify for APG&E plans.
